# Detect an unexpected recipe change
Attach your Day 1 Lakehouse as the default. Use your own lab ID in place of `lab07` in the file path and every output table name. For a Lakehouse without schemas, omit `dbo.` throughout. The input is a fixed synthetic batch, not a live production feed. Run the cells in order.

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW process_input_file USING text
OPTIONS (path 'Files/process_lab07/process_batch.json', wholetext 'true')

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW process_batch AS
SELECT from_json(
    CASE WHEN get_json_object(value, '$.encoding') = 'base64'
         THEN decode(unbase64(get_json_object(value, '$.content')), 'UTF-8')
         ELSE value END,
    'STRUCT<recipePlans:ARRAY<STRUCT<PlantId:STRING,MachineId:STRING,LineName:STRING,ExpectedRecipe:STRING>>,approvedChanges:ARRAY<STRUCT<RunId:STRING,ApprovedRecipe:STRING>>,runs:ARRAY<STRUCT<RunId:STRING,PlantId:STRING,MachineId:STRING,RunDateUtc:STRING,ActualRecipe:STRING>>>',
    map('mode', 'FAILFAST')
) AS payload
FROM process_input_file

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW recipe_plans AS
SELECT inline(payload.recipePlans) FROM process_batch

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW approved_changes AS
SELECT inline(payload.approvedChanges) FROM process_batch

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW process_runs_raw AS
SELECT inline(payload.runs) FROM process_batch

## Try the rule before scrolling to the reference SQL
Join each run to `recipe_plans` on both plant and machine, and to `approved_changes` by `RunId`. A missing plan is invalid data, not a recipe anomaly. A mismatch is allowed only when that exact run has approval for the actual recipe. Write your own `CASE` expression in your notes, then compare it to the reference cell below.

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW assessed_process_runs AS
SELECT r.RunId, CONCAT(r.PlantId, ':', r.MachineId) AS MachineKey,
       r.PlantId, r.MachineId, plant.PlantName, p.LineName, r.RunDateUtc,
       p.ExpectedRecipe, r.ActualRecipe, a.ApprovedRecipe,
       CASE
           WHEN r.RunId IS NULL OR r.RunDateUtc IS NULL OR plant.PlantId IS NULL
                OR r.ActualRecipe IS NULL OR p.ExpectedRecipe IS NULL THEN 'InvalidInput'
           WHEN r.ActualRecipe = p.ExpectedRecipe THEN 'Normal'
           WHEN r.ActualRecipe = a.ApprovedRecipe THEN 'ApprovedChange'
           ELSE 'Anomaly'
       END AS RunStatus
FROM process_runs_raw r
LEFT JOIN recipe_plans p ON r.PlantId = p.PlantId AND r.MachineId = p.MachineId
LEFT JOIN dbo.plants plant ON r.PlantId = plant.PlantId
LEFT JOIN approved_changes a ON r.RunId = a.RunId

In [ ]:
%%sql
SELECT assert_true(
    COUNT(*) > 0 AND COUNT(*) = COUNT(DISTINCT RunId)
    AND COUNT(CASE WHEN RunStatus = 'InvalidInput' THEN 1 END) = 0,
    'Empty batch, duplicate RunId, or a run missing required data or a recipe plan. Check the input before publishing.'
) AS PassedValidation
FROM assessed_process_runs

## Publish a repeatable workshop snapshot
These three cells replace only your lab-ID-specific output tables. They never modify `plants`, `machine_events`, or the raw input. This is safe to replay with the same fixture; an incremental production implementation would preserve history and merge by stable keys.

In [ ]:
%%sql
CREATE OR REPLACE TABLE dbo.process_machines_lab07 USING DELTA AS
SELECT DISTINCT MachineKey, PlantId, MachineId, PlantName, LineName
FROM assessed_process_runs

In [ ]:
%%sql
CREATE OR REPLACE TABLE dbo.process_runs_lab07 USING DELTA AS
SELECT * FROM assessed_process_runs

In [ ]:
%%sql
CREATE OR REPLACE TABLE dbo.process_anomalies_lab07 USING DELTA AS
SELECT CONCAT(RunId, ':recipe_mismatch') AS AnomalyId, RunId, MachineKey,
       PlantId, MachineId, RunDateUtc, ExpectedRecipe, ActualRecipe,
       'Recipe differs from the plan without an approved change for this run' AS Reason
FROM assessed_process_runs
WHERE RunStatus = 'Anomaly'

In [ ]:
%%sql
SELECT RunStatus, COUNT(*) AS RunCount FROM dbo.process_runs_lab07 GROUP BY RunStatus ORDER BY RunStatus

In [ ]:
%%sql
SELECT AnomalyId, MachineId, RunDateUtc, ExpectedRecipe, ActualRecipe, Reason FROM dbo.process_anomalies_lab07